# 01 — Extracción

**Fase 1 del proyecto ISO 20022 en Redes DLT.**

Se conecta a las APIs públicas de **XRPL**, **Stellar** y **Hedera** para extraer transacciones históricas con memo no vacío, y en paralelo descarga indicadores macroeconómicos de **FRED**. Los datos crudos se guardan en `data/raw/` (por red) y `data/external/` (FRED).

> **Nota XRPL:** el RPC público no indexa por fecha. La extracción recorre un rango de ledgers recientes acotado por `XRPL_MAX_LEDGERS` (ajustable). Stellar y Hedera sí permiten cortar por fecha mediante paginación.

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de archivos

# 2. Imports

In [ ]:
import os
import json
import time
from datetime import datetime, date, timezone
from typing import Any

import requests
import httpx

# 3. Paths y Constantes

In [ ]:
# Parámetros globales (ver SKILL.md)
RANGO_INICIO: str = "2025-01-01"           # YTD
RANGO_FIN: str = date.today().isoformat()  # fecha ejecución
GRANULARIDAD: str = "1D"                   # diaria
FILTRO_MEMO: bool = True                   # solo memo no vacío

# Límites de seguridad para las llamadas paginadas
TIMEOUT: int = 30            # segundos por request
MAX_PAGINAS: int = 50        # tope de páginas por red
PAGINA_STELLAR: int = 200    # registros por página (máx)
PAGINA_HEDERA: int = 100     # registros por página
XRPL_MAX_LEDGERS: int = 500  # tope de ledgers a recorrer

# Endpoints públicos
XRPL_RPC: str = "https://xrplcluster.com/"
STELLAR_HORIZON: str = "https://horizon.stellar.org"
HEDERA_MIRROR: str = "https://mainnet-public.mirrornode.hedera.com"
FRED_API: str = ("https://api.stlouisfed.org/fred/"
                 "series/observations")

# FRED requiere API key gratuita (https://fred.stlouisfed.org)
FRED_API_KEY: str = os.environ.get("FRED_API_KEY", "")

# Series macroeconómicas de FRED a extraer
FRED_SERIES: dict[str, str] = {
    "STLFSI4": "St. Louis Fed Financial Stress Index",
    "DFF": "Federal Funds Effective Rate",
    "SOFR": "Secured Overnight Financing Rate",
    "RRPONTSYD": "Overnight Reverse Repurchase Agreements",
}

# Paths de salida (data/raw/ y data/external/)
BASE_DIR: str = "tesis"
DIR_RAW: dict[str, str] = {
    "xrpl": os.path.join(BASE_DIR, "data", "raw", "xrpl"),
    "stellar": os.path.join(BASE_DIR, "data", "raw", "stellar"),
    "hedera": os.path.join(BASE_DIR, "data", "raw", "hedera"),
}
DIR_EXTERNAL: str = os.path.join(BASE_DIR, "data", "external")

# 4. Funciones

In [ ]:
def guardar_json(datos: Any, ruta: str) -> None:
    """
    Guarda una estructura como JSON si SAVE_DATA está activo.

    Args:
        datos: objeto serializable a JSON.
        ruta: path de destino del archivo.
    """
    if not SAVE_DATA:
        if DEBUG:
            print(f"[DEBUG] SAVE_DATA=False, no guardo {ruta}")
        return
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")

In [ ]:
def xrpl_rpc(metodo: str, params: dict) -> dict:
    """
    Ejecuta una llamada JSON-RPC contra el nodo público XRPL.

    Args:
        metodo: nombre del método RPC (p. ej. 'ledger').
        params: parámetros del método.

    Returns:
        Diccionario 'result' de la respuesta, o {} si falla.
    """
    payload = {"method": metodo, "params": [params]}
    try:
        r = requests.post(XRPL_RPC, json=payload,
                          timeout=TIMEOUT)
        r.raise_for_status()
        return r.json().get("result", {})
    except (requests.RequestException, ValueError) as e:
        if DEBUG:
            print(f"[DEBUG] error XRPL RPC {metodo}: {e}")
        return {}


def xrpl_ledger_validado() -> int:
    """
    Devuelve el índice del último ledger validado.

    Returns:
        Índice de ledger, o 0 si no se pudo obtener.
    """
    res = xrpl_rpc("ledger", {"ledger_index": "validated"})
    indice = res.get("ledger", {}).get("ledger_index", 0)
    return int(indice) if indice else 0


def xrpl_txs_de_ledger(indice: int) -> list[dict]:
    """
    Recupera las transacciones expandidas de un ledger.

    Args:
        indice: índice del ledger a consultar.

    Returns:
        Lista de transacciones (dicts) del ledger.
    """
    res = xrpl_rpc("ledger", {
        "ledger_index": indice,
        "transactions": True,
        "expand": True,
    })
    return res.get("ledger", {}).get("transactions", [])


def xrpl_extraer(max_ledgers: int) -> list[dict]:
    """
    Recorre ledgers hacia atrás y recupera txs con memo.

    Args:
        max_ledgers: cantidad máxima de ledgers a recorrer.

    Returns:
        Lista de transacciones XRPL con Memos no vacío.
    """
    fin = xrpl_ledger_validado()
    if not fin:
        return []
    txs_memo: list[dict] = []
    for indice in range(fin, fin - max_ledgers, -1):
        for tx in xrpl_txs_de_ledger(indice):
            cuerpo = tx.get("tx_json", tx)
            if not FILTRO_MEMO or cuerpo.get("Memos"):
                txs_memo.append(tx)
        time.sleep(0.05)  # cortesía con el nodo público
    if DEBUG:
        print(f"[DEBUG] XRPL: {len(txs_memo)} txs con memo "
              f"en {max_ledgers} ledgers")
    return txs_memo

In [ ]:
def stellar_extraer(inicio: str,
                    max_paginas: int) -> list[dict]:
    """
    Pagina Horizon y recupera txs con memo hasta 'inicio'.

    Args:
        inicio: fecha mínima (ISO) de created_at a incluir.
        max_paginas: tope de páginas a recorrer.

    Returns:
        Lista de transacciones Stellar con memo no vacío.
    """
    url = (f"{STELLAR_HORIZON}/transactions"
           f"?order=desc&limit={PAGINA_STELLAR}"
           f"&include_failed=false")
    txs_memo: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for _ in range(max_paginas):
            try:
                r = cliente.get(url)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error Stellar: {e}")
                break
            registros = datos.get(
                "_embedded", {}).get("records", [])
            if not registros:
                break
            for tx in registros:
                if (not FILTRO_MEMO
                        or tx.get("memo_type") != "none"):
                    txs_memo.append(tx)
            if registros[-1].get("created_at", "") < inicio:
                break
            url = datos.get(
                "_links", {}).get("next", {}).get("href", "")
            if not url:
                break
    if DEBUG:
        print(f"[DEBUG] Stellar: {len(txs_memo)} txs con memo")
    return txs_memo

In [ ]:
def hedera_extraer(inicio: str,
                   max_paginas: int) -> list[dict]:
    """
    Pagina el Mirror Node y recupera txs con memo.

    Args:
        inicio: fecha mínima (ISO) a incluir.
        max_paginas: tope de páginas a recorrer.

    Returns:
        Lista de transacciones Hedera con memo_base64.
    """
    ruta = (f"/api/v1/transactions?limit={PAGINA_HEDERA}"
            f"&order=desc")
    inicio_epoch = datetime.fromisoformat(inicio).replace(
        tzinfo=timezone.utc).timestamp()
    txs_memo: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for _ in range(max_paginas):
            try:
                r = cliente.get(HEDERA_MIRROR + ruta)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error Hedera: {e}")
                break
            registros = datos.get("transactions", [])
            if not registros:
                break
            for tx in registros:
                if not FILTRO_MEMO or tx.get("memo_base64"):
                    txs_memo.append(tx)
            ultimo = registros[-1].get(
                "consensus_timestamp", "0")
            if float(ultimo) < inicio_epoch:
                break
            ruta = datos.get("links", {}).get("next", "")
            if not ruta:
                break
    if DEBUG:
        print(f"[DEBUG] Hedera: {len(txs_memo)} txs con memo")
    return txs_memo

In [ ]:
def fred_serie(series_id: str, inicio: str,
               fin: str) -> list[dict]:
    """
    Descarga las observaciones de una serie de FRED.

    Args:
        series_id: identificador de la serie (p. ej. 'DFF').
        inicio: fecha inicial (ISO).
        fin: fecha final (ISO).

    Returns:
        Lista de observaciones (fecha/valor), o [] si falla.
    """
    if not FRED_API_KEY:
        if DEBUG:
            print("[DEBUG] falta FRED_API_KEY, omito FRED")
        return []
    params = {
        "series_id": series_id,
        "observation_start": inicio,
        "observation_end": fin,
        "api_key": FRED_API_KEY,
        "file_type": "json",
    }
    try:
        r = requests.get(FRED_API, params=params,
                         timeout=TIMEOUT)
        r.raise_for_status()
        return r.json().get("observations", [])
    except (requests.RequestException, ValueError) as e:
        if DEBUG:
            print(f"[DEBUG] error FRED {series_id}: {e}")
        return []


def fred_extraer(inicio: str,
                 fin: str) -> dict[str, list[dict]]:
    """
    Descarga todas las series definidas en FRED_SERIES.

    Args:
        inicio: fecha inicial (ISO).
        fin: fecha final (ISO).

    Returns:
        Diccionario {series_id: observaciones}.
    """
    resultado: dict[str, list[dict]] = {}
    for series_id in FRED_SERIES:
        obs = fred_serie(series_id, inicio, fin)
        resultado[series_id] = obs
        if DEBUG:
            print(f"[DEBUG] FRED {series_id}: {len(obs)} obs")
    return resultado

# 5. Proceso

In [ ]:
# Sello temporal (UTC) para nombrar los archivos crudos
sello: str = datetime.now(timezone.utc).strftime("%Y%m%d")

# --- XRPL ---
xrpl_txs = xrpl_extraer(XRPL_MAX_LEDGERS)
guardar_json(
    xrpl_txs,
    os.path.join(DIR_RAW["xrpl"], f"xrpl_{sello}.json"),
)

# --- Stellar ---
stellar_txs = stellar_extraer(RANGO_INICIO, MAX_PAGINAS)
guardar_json(
    stellar_txs,
    os.path.join(DIR_RAW["stellar"], f"stellar_{sello}.json"),
)

# --- Hedera ---
hedera_txs = hedera_extraer(RANGO_INICIO, MAX_PAGINAS)
guardar_json(
    hedera_txs,
    os.path.join(DIR_RAW["hedera"], f"hedera_{sello}.json"),
)

# --- FRED (indicadores macro) ---
fred_datos = fred_extraer(RANGO_INICIO, RANGO_FIN)
guardar_json(
    fred_datos,
    os.path.join(DIR_EXTERNAL, f"fred_{sello}.json"),
)

if DEBUG:
    print("\n[DEBUG] Resumen de extracción")
    print(f"  XRPL:    {len(xrpl_txs)} txs")
    print(f"  Stellar: {len(stellar_txs)} txs")
    print(f"  Hedera:  {len(hedera_txs)} txs")
    print(f"  FRED:    {len(fred_datos)} series")